# Part 4: Spatial Analysis and Geographic Mapping

Statistical tables only tell part of the story. In historical demography, geography matters immensely. This notebook leverages `geopandas` to map the spatial distribution of Catholics and the geographic footprint of the fertility response, allowing us to visually inspect for spatial clustering or omitted regional variables.


### 1. Setup and Loading the Shapefile
We load the HGIS German Empire shapefile and merge it with our panel dataset.


In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Enable autoreload for development
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Paths
DATA_RAW = project_root / "data" / "raw"
DATA_PROCESSED = project_root / "data" / "processed"
OUTPUTS = project_root / "outputs" / "figures"
OUTPUTS.mkdir(exist_ok=True, parents=True)

print("Setup complete. Outputs will be saved to:", OUTPUTS)

from src.visualization.maps import (
    load_prussia_shapefile, map_catholic_share, map_fertility_change,
    map_polish_german_provinces, map_kulturkampf_residuals,
)

panel = pd.read_parquet(DATA_PROCESSED / "analysis_panel.parquet")

shp_path = DATA_RAW / "German_Empire_1871_v.1.0.shp"
gdf = load_prussia_shapefile(shp_path)
print(f"Loaded {len(gdf)} Prussian counties from shapefile")



### 2. Mapping the Treatment: Catholic Share
First, we visualize our primary independent variable. This map highlights the deep religious divide in Prussia: the heavily Catholic Rhineland in the West and Polish territories in the East, contrasted against the Protestant Prussian core.


In [ ]:
fig, ax = map_catholic_share(
    gdf, panel,
    savepath=str(OUTPUTS / "map1_catholic_share.png"),
)
plt.show()



### 3. Mapping the Outcome: Fertility Change
Next, we calculate the naive difference in fertility between the pre-Kulturkampf era (1868-1872) and the post era (1878-1882) and map it. Darker areas experienced the largest fertility increases.


In [ ]:
fig, ax = map_fertility_change(
    gdf, panel,
    pre_years=(1868, 1872),
    post_years=(1878, 1882),
    savepath=str(OUTPUTS / "map2_fertility_change.png"),
)
plt.show()



### 4. Polish vs. German Catholic Regions
To aid in our heterogeneity analysis from Notebook 3, we map out the specific Polish-majority provinces versus the rest of Prussia. This highlights the geographic overlap between Catholicism and Polish ethnicity in the Eastern provinces.


In [ ]:
fig, ax = map_polish_german_provinces(
    gdf, panel,
    savepath=str(OUTPUTS / "map3_polish_german.png"),
)
plt.show()



### 5. Residual Mapping
Finally, we map the residuals from our Difference-in-Differences model. If there is strong spatial autocorrelation in the residuals (e.g., all prediction errors are clustered in one specific region), it suggests an omitted variable bias tied to geography. A random scatter of residuals indicates a well-specified model.


In [ ]:
fig, ax = map_kulturkampf_residuals(
    gdf, panel,
    pre_years=(1868, 1872),
    post_years=(1873, 1878),
    savepath=str(OUTPUTS / "map4_residuals.png"),
)
plt.show()

